##### 常數

In [1]:
IMG_PATH = "../../../../resources/pics/A2.jpg"  # 圖片路徑
MODEL = "./models/BlazePose/pose_landmarker_full.task"  # 模型路徑

#### 套件

In [2]:
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt

from mediapipe.tasks.python.core.base_options import BaseOptions
from mediapipe.tasks.python.vision.core.vision_task_running_mode import (
    VisionTaskRunningMode,
)
from mediapipe.tasks.python.vision.pose_landmarker import (
    PoseLandmarker,
    PoseLandmarkerOptions,
)

In [3]:
from mpl_toolkits.mplot3d.axes3d import Axes, Axes3D
from mediapipe.tasks.python.vision.pose_landmarker import PoseLandmarkerResult

In [4]:
from src.mediapipe_lib.base import PoseResult, ResultAnalyzer
from src.draw_template import (
    draw_debug_in_img,
    draw_kpt_position_in_img,
    get_bones_plot_axes,
    get_staggered_angle_axes,
    get_hand_center_to_gravity_axes,
)
from src.utils.plot_painter import set_data_range, set_plot_labels, plot_to_opencv_img

#### 忽略警告

In [5]:
import warnings

warnings.filterwarnings("ignore")

#### 函式

##### 設定圖表座標資訊

In [6]:
def set_axes_info(is_3d: bool, ax: Axes | Axes3D) -> Axes | Axes3D:
    """設定圖表座標資訊

    Args:
        is_3d (bool): 是否為 3D 姿勢
        ax (Axes | Axes3D): 圖表座標

    Returns:
        Axes | Axes3D: 圖表座標
    """

    # 設定圖表座標資訊
    if not is_3d:
        set_plot_labels("x", "y", ax=ax)  # 設定標籤
        set_data_range([0, 1], [1, 0], ax=ax)  # 設定資料範圍
    else:
        set_plot_labels("x", "z", "y", ax)  # 設定標籤
        set_data_range([-1, 1], [-1, 1], [0, 2], ax)  # 設定資料範圍

    return ax

##### 繪製分析結果圖表

In [7]:
def draw_result_plot(
    result: PoseLandmarkerResult,
    figsize: tuple[float, float] = (6.4, 4.8),
    is_3d: bool = False,
) -> cv2.typing.MatLike:
    """繪製分析結果圖表

    Args:
        result (PoseLandmarkerResult): 姿勢結果
        figsize (tuple[float, float], optional): 圖表大小. Defaults to (6.4, 4.8).
        is_3d (bool, optional): 是否為 3D 姿勢. Defaults to False.

    Returns:
        cv2.typing.MatLike: 分析結果圖片
    """
    # 建立分析器
    analyzer = ResultAnalyzer(PoseResult(result))

    # 建立圖表
    fig = plt.figure(figsize=figsize)
    ax: Axes3D = fig.add_subplot(projection="3d")

    # 設定圖表座標資訊
    set_axes_info(is_3d, ax)

    # 繪製表格
    if len(result.pose_world_landmarks) > 0:
        ax.set_title(
            "Hand to gravity distance:"
            + f"{analyzer.get_a_hand_to_gravity_dist(True, is_3d):.2f}, "
            + f"{analyzer.get_a_hand_to_gravity_dist(False,is_3d):.2f}, "
            + f"{analyzer.get_two_hands_center_to_gravity_dist(is_3d):.2f}"
        )
        get_bones_plot_axes(result, is_3d=is_3d, ax=ax)
        get_hand_center_to_gravity_axes(result, is_3d, ax)
        ax.legend()
    else:
        ax.set_title("NO DATA")

    # 將圖表轉換成圖片格式
    rgb = cv2.cvtColor(plot_to_opencv_img(fig), cv2.COLOR_RGBA2RGB)

    plt.close(fig)  # 關閉圖表

    return rgb

#### 開始測試

In [8]:
# 模型設定
options = PoseLandmarkerOptions(
    base_options=BaseOptions(MODEL), running_mode=VisionTaskRunningMode.IMAGE
)

# 使用 3D 模型
is_3d = True

In [16]:
# 圖片資料
mp_img = mp.Image.create_from_file(IMG_PATH)
np_img: np.ndarray = mp_img.numpy_view()
cv_img = cv2.cvtColor(np_img, cv2.COLOR_RGB2BGR)

# 圖片格式
h, w, _ = cv_img.shape

# 表格大小
plot_size = ((w / 50), (h / 100))

# CV 輸出畫面
cv_view = np.zeros((100, 100, 3))

# 使用模型
with PoseLandmarker.create_from_options(options) as landmark:
    # 取得姿勢結果
    result = landmark.detect(mp_img)

    # 繪製圖片關鍵點和偵錯資訊
    img_view = draw_kpt_position_in_img(result, cv_img)
    img_view = draw_debug_in_img(result, img_view, is_3d)
    # 繪製分析圖表
    plot_view = draw_result_plot(result, plot_size, is_3d)

    # 組合圖片
    cv_view: cv2.typing.MatLike = np.concatenate((img_view, plot_view), axis=1)

cv2.imshow("debug", cv_view)
cv2.waitKey(0)
cv2.destroyAllWindows()